# The Agent Loop as a Graph

**What you will build:** the ReAct loop from 0.3 — but as an explicit LangGraph with a real cycle. This is exactly what `create_agent` builds for you (2.0); building it by hand shows you the machine and prepares you for customizing it.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/22_agent_loop_as_graph.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — LangChain + LangGraph on OpenRouter. Run once.
!pip install -q "langchain>=1.3,<2.0" "langgraph>=1.2,<2.0" "langchain-openai>=1.3,<2.0"

import os, getpass
from langchain_openai import ChatOpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
llm = ChatOpenAI(model=MODEL_NAME, base_url="https://openrouter.ai/api/v1",
                 api_key=os.environ["OPENROUTER_API_KEY"])
print("Ready:", MODEL_NAME)

## The plan: two nodes and a loop

The ReAct loop is: call the model; if it asked for a tool, run the tool and go back to the model; otherwise stop. As a graph that's two nodes with a **conditional edge that can loop back**:

```
        ┌──────────────────────────────┐
        ▼                              │ (tool was called)
  START ─▶ model ──▶ tool calls? ──────┘
                        │ no
                        ▼
                       END
```

In [ ]:
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    fake = {"bilbao": "18C, light rain", "madrid": "31C, sunny"}
    return fake.get(city.lower(), "20C, clear")

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv, ast.USub: operator.neg}
def safe_eval(expr):                       # same helper as notebook 0.3 -- no eval(), so 9**9**9 can't hang the kernel
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.operand))
        raise ValueError('unsupported')
    return ev(ast.parse(expr, mode='eval'))

@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression like '12 * (3 + 4)'."""
    try:
        return str(safe_eval(expression))
    except Exception:
        return "Error: only +, -, *, / and parentheses are supported."

tools = [get_weather, calculator]
llm_with_tools = llm.bind_tools(tools)   # tells the model which tools exist

`MessagesState` is a built-in state with one field, `messages`, using the `add_messages` reducer so the list grows correctly (that's the reducer from 2.1). The model node calls the tool-aware LLM; the router checks the last message for tool calls.

In [ ]:
def call_model(state: MessagesState) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

def should_continue(state: MessagesState) -> str:
    last = state["messages"][-1]
    return "tools" if last.tool_calls else END      # loop to tools, or stop

builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_node("tools", ToolNode(tools))          # runs whatever tools the model asked for

builder.add_edge(START, "model")
builder.add_conditional_edges("model", should_continue, {"tools": "tools", END: END})
builder.add_edge("tools", "model")                   # <-- the cycle: back to the model

graph = builder.compile()

In [ ]:
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))     # PNG via mermaid.ink (needs internet)
except Exception:
    print("Mermaid image unavailable, showing text instead:\n")
    print(graph.get_graph().draw_mermaid())                  # pure text (Mermaid DSL), no extra deps

There's the loop, drawn: `model -> tools -> model` until no tool is requested. Run it on the two-tool question:

In [ ]:
out = graph.invoke({"messages": [{"role": "user",
        "content": "What's the weather in Madrid, and what is 231 * 17?"}]})
for m in out["messages"]:
    m.pretty_print()

Read the printed messages: user question, an AI message with tool calls, the tool results, then the final AI answer — the exact sequence you appended by hand in 0.3, now managed by the graph. `create_agent` from 2.0 is just this graph, prebuilt. Now you can modify it — add a node, change the routing, insert a guard — which is the whole reason to drop down to LangGraph.

## Watch it loop, live: `graph.stream()`

`invoke()` hands you the final state; in production you almost always want the steps *as they happen* — a UI shows progress, a log records the path, you see the loop looping. That's `stream()`, and `stream_mode="updates"` yields, for every node execution, what that node just wrote:

In [ ]:
for chunk in graph.stream(
    {"messages": [{"role": "user", "content": "What's the weather in Bilbao, and what is 88 * 11?"}]},
    stream_mode="updates",
):
    for node, update in chunk.items():
        last = update["messages"][-1]
        summary = str(last.content)[:60] or f"tool_calls: {[tc['name'] for tc in last.tool_calls]}"
        print(f"[{node:6s}] {summary}")

Read the `[model]` / `[tools]` alternation — that's the cycle from the drawing, executing in real time. Three `stream_mode` values cover most needs: `"updates"` (what each node wrote — this one), `"values"` (the full state after each step), and `"messages"` (token-by-token LLM output, for chat UIs — you'll use it in 3.0). From here on, whenever a graph loops (2.5's reflection, 2.7's retrieval cycle), we stream it — watching iterations beats reading a final blob.

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Add a guard node.** Insert a node between `START` and `model` that rejects empty questions, and route to `END` if the input is blank. **Done when:** `graph.invoke({"messages": [{"role": "user", "content": ""}]})` comes back with your rejection message and zero LLM calls.
2. **Cap the loop.** Call `graph.invoke(..., {"recursion_limit": 3})` with the two-tool question and watch `GraphRecursionError` trip. Careful with the units: the limit counts **super-steps** (every node execution — `model` and `tools` each count one), *not* conversational turns. Our two-tool run needs `model → tools → model` = 3 steps minimum, 5 if the model calls the tools one at a time. **Done when:** you can predict the minimum limit for a given question before running it.
3. **Swap the model** in the setup cell to a Claude or GPT id and confirm the identical graph still runs.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**1 — Guard node:**

```python
def guard(state: MessagesState) -> dict:
    return {}                                   # writes nothing; it only routes

def check_input(state: MessagesState) -> str:
    text = state["messages"][-1].content if state["messages"] else ""
    return "model" if str(text).strip() else END

builder = StateGraph(MessagesState)
builder.add_node("guard", guard)
builder.add_node("model", call_model)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "guard")
builder.add_conditional_edges("guard", check_input, {"model": "model", END: END})
builder.add_conditional_edges("model", should_continue, {"tools": "tools", END: END})
builder.add_edge("tools", "model")
guarded = builder.compile()
```

Your first structural edit of the loop — the thing 1.x couldn't give you. (Returning `{}` and routing in the conditional keeps the guard side-effect-free; you could also have it write a rejection message into `messages`.)

**2 —** With parallel tool calls: `model(1) → tools(2) → model(3)` — limit 3 passes, 2 trips. If the model requests tools one at a time: 5. The formula: `2 × tool-rounds + 1`. It's 0.3's `max_turns` *in different units* — budget steps, not turns.

**3 —** One string. If the new model formats tool arguments differently, `ToolNode` still handles it — the messages protocol (0.3) is doing the compatibility work.
::::

::::{dropdown} 🔧 Common issues
:color: info

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| **`GraphRecursionError`** | The model keeps requesting tools; the cycle never ends | Raise/lower `recursion_limit`; check the tool returns something the model treats as done |
| `recursion_limit` trips earlier than expected | It counts **super-steps** (each node run), not turns | Budget `2 × tool-rounds + 1` steps for this loop |
| **Tools never run** | The conditional edge doesn't reach the `tools` node | Verify `should_continue` returns `"tools"` when `last.tool_calls` is truthy |
| **`messages` overwritten, not appended** | Missing the `add_messages` reducer | Use `MessagesState`, or `Annotated[list, add_messages]` on the field |
::::

**What's next:** in **2.3** we give the graph real **memory** with a checkpointer — persistent, per-conversation state that survives across calls.